In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

In [2]:
def minmax_norm(s):
    s = s.astype(float)
    return (s - s.min()) / (s.max() - s.min())

In [3]:
alias_file = "9606.protein.aliases.v12.0.txt"
links_file = "9606.protein.links.detailed.v12.0.txt"

In [4]:
df = pd.read_csv('all_proteins_max_score_matrix_cleaned.csv',index_col=0)

In [5]:
star = (
    df.stack()
      .reset_index()
      .iloc[:, :2]  # keep only protein + TRN
)

star.columns = ["protein", "trn"]

star["protein"] = star["protein"].astype(str).str.strip().str.rstrip(".")
star["trn"] = star["trn"].astype(str).str.strip().str.rstrip(".")

In [6]:
aliases = pd.read_csv(alias_file, sep="\t", low_memory=False)
aliases = aliases[aliases["#string_protein_id"].str.startswith("9606.")].copy()
aliases["alias"] = aliases["alias"].astype(str).str.strip()

In [7]:
preferred_sources = [
    "Ensembl_HGNC",
    "Ensembl_HGNC_symbol",
    "BLAST_UniProt_GN",
    "UniProt_GN",
    "BioMart_HUGO"
]
aliases["source_rank"] = np.where(
    aliases["source"].isin(preferred_sources), 0, 1
)
aliases = aliases.sort_values(["alias", "source_rank"])
alias_to_string = aliases.drop_duplicates("alias").set_index("alias")["#string_protein_id"]

In [8]:
star["protein_string"] = star["protein"].map(alias_to_string)
star["trn_string"] = star["trn"].map(alias_to_string)

In [9]:
star["pair_key"] = star.apply(
    lambda r: tuple(sorted([r["protein_string"], r["trn_string"]]))
    if pd.notna(r["protein_string"]) and pd.notna(r["trn_string"])
    else np.nan,
    axis=1
)

pairs_needed = set(star["pair_key"].dropna())

In [10]:
usecols = ["protein1", "protein2", "combined_score", "experimental", "database", "textmining"]
hits = []

for chunk in pd.read_csv(
    links_file,
    sep=" ",
    usecols=usecols,
    chunksize=2_000_000
):
    chunk["pair_key"] = list(zip(
        np.minimum(chunk["protein1"], chunk["protein2"]),
        np.maximum(chunk["protein1"], chunk["protein2"])
    ))
    
    sub = chunk[chunk["pair_key"].isin(pairs_needed)].copy()
    if len(sub) > 0:
        hits.append(sub)

if len(hits) > 0:
    links_sub = pd.concat(hits, ignore_index=True)
    links_sub = links_sub.drop_duplicates("pair_key")
else:
    links_sub = pd.DataFrame(columns=usecols + ["pair_key"])

In [11]:
star_out = star.merge(
    links_sub[["pair_key", "combined_score", "experimental", "database", "textmining"]],
    on="pair_key",
    how="left"
)

# helpful flags
star_out["has_textmining"] = star_out["textmining"].fillna(0) > 0
star_out["has_experimental_evidence"] = star_out["experimental"].fillna(0) > 0
star_out["has_database_evidence"] = star_out["database"].fillna(0) > 0

In [12]:
cols_to_norm = ["combined_score", "experimental", "database", "textmining"]

star_out[[f"{c}_norm" for c in cols_to_norm]] = star_out[cols_to_norm].apply(minmax_norm)

In [13]:
star_out = star_out.fillna(0)

In [14]:
star_out.to_csv('all_scores.csv')